# Semantic Boundary Analysis

This notebook visualizes how omnichunk's `detect_topic_shifts` scores each
sentence gap and where it places topic boundaries. It runs entirely offline
(TF-IDF coherence scoring; no embedding API required).

In [ ]:
from omnichunk.semantic.tfidf import detect_topic_shifts

# (a) A synthetic document with three disjoint-vocabulary topics.
cooking = ["We diced onions, garlic, and tomatoes for the simmering sauce."] * 5
astronomy = ["The galaxy spins around a supermassive black hole at its core."] * 5
finance = ["Quarterly revenue beat the analyst earnings forecast this year."] * 5
sentences = cooking + astronomy + finance
true_boundaries = [len(cooking) - 1, len(cooking) + len(astronomy) - 1]
print(f"{len(sentences)} sentences; true boundaries at gaps {true_boundaries}")

In [ ]:
# (b) Compute boundaries and per-gap coherence scores.
shifts, scores = detect_topic_shifts(
    sentences, method="adaptive", window_size=3, min_shift_gap=2, return_scores=True
)
print("detected boundaries (gap indices):", list(shifts))
print("score curve length:", len(scores))

In [ ]:
# (c) Plot the similarity score curve with boundary markers.
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(len(scores)), scores, marker="o", label="coherence score")
    for b in shifts:
        ax.axvline(b, color="red", linestyle="--", alpha=0.7)
    for t in true_boundaries:
        ax.axvline(t, color="green", linestyle=":", alpha=0.5)
    ax.set_xlabel("sentence gap index")
    ax.set_ylabel("TF-IDF window cosine similarity")
    ax.set_title("Topic-shift coherence curve (red=detected, green=ground truth)")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed; score curve:")
    for gap, score in enumerate(scores):
        mark = " <-- boundary" if gap in set(shifts) else ""
        print(f"gap[{gap:>3}] {score:.4f}{mark}")